In [18]:
import nltk
# Download the corpus and necessary tokenizers/resources
nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4') # Needed for lemmatization
nltk.download('punkt_tab') # Required for gutenberg.sents() in some configurations

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [19]:
# State selected choice
print("Selected Corpus: Option A - Gutenberg 'austen-emma.txt'")

Selected Corpus: Option A - Gutenberg 'austen-emma.txt'


## Part A - Text preprocessing

Goal: Prepare a clean token stream and justify your choices.

In [20]:
from nltk.corpus import gutenberg

# 1. Load the corpus
corpus_name = 'austen-emma.txt'
raw_text = gutenberg.raw(corpus_name)
raw_tokens = gutenberg.words(corpus_name)

# 2. Print initial statistics
print("--- Part A1: Initial Stats ---")
print(f"Total number of characters: {len(raw_text)}")
print(f"Total number of tokens BEFORE preprocessing: {len(raw_tokens)}")

--- Part A1: Initial Stats ---
Total number of characters: 887071
Total number of tokens BEFORE preprocessing: 192427


In [21]:
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import Counter

def preprocess_text(tokens, remove_stopwords=True, use_lemmatization=True):
    """
    Preprocesses a list of tokens:
    1. Lowercase
    2. Remove punctuation
    3. (Optional) Remove stopwords
    4. (Optional) Lemmatize
    """

    # Initialize tools
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    clean_tokens = []

    for token in tokens:
        # 1. Lowercase
        token = token.lower()

        # 2. Remove Punctuation
        # We check if the token is ONLY punctuation (e.g., "." or "--")
        if all(char in string.punctuation for char in token):
            continue

        # 3. Stopword Removal (Optional)
        if remove_stopwords and token in stop_words:
            continue

        # 4. Lemmatization (Optional)
        if use_lemmatization:
            token = lemmatizer.lemmatize(token)

        clean_tokens.append(token)

    return clean_tokens

# Run the preprocessing
# Note: passing raw_tokens from A1
processed_tokens = preprocess_text(raw_tokens, remove_stopwords=True, use_lemmatization=True)

In [22]:
# Calculate statistics
total_clean_tokens = len(processed_tokens)
vocabulary_size = len(set(processed_tokens)) # Set removes duplicates to find unique types

# Get Top 20 using Counter
token_counts = Counter(processed_tokens)
top_20 = token_counts.most_common(20)

print("\n--- Part A2: Post-Preprocessing Stats ---")
print(f"Total number of tokens: {total_clean_tokens}")
print(f"Vocabulary size: {vocabulary_size}")
print("\nTop 20 most frequent tokens:")
for word, count in top_20:
    print(f"{word}: {count}")


--- Part A2: Post-Preprocessing Stats ---
Total number of tokens: 73529
Vocabulary size: 6435

Top 20 most frequent tokens:
mr: 1852
emma: 865
could: 837
would: 820
miss: 600
must: 567
harriet: 506
much: 486
said: 484
thing: 460
one: 456
weston: 448
every: 435
think: 406
well: 401
knightley: 389
elton: 385
know: 367
little: 359
never: 358


### Reflection

Preprocessing choices involve trade-offs between dimensionality reduction and information loss. By lowercasing and lemmatizing, I reduced the vocabulary size, which creates denser vector representations and reduces sparsity for bag-of-words models. However, removing stopwords, while beneficial for topic classification (reducing noise), can be detrimental for n-gram language models. A language model needs stopwords (like 'the' or 'of') to predict syntactically correct sentences, while removing them destroys the natural flow of text required for next-word prediction.

## Part B - Text representation

Goal: Compare Bag-of-Words and TF-IDF representations and interpret the results.

In [23]:
# 1. Get raw tokens again (to ensure we have the full structure)
raw_tokens = list(gutenberg.words('austen-emma.txt'))

# 2. Split into chunks of 500 tokens
chunk_size = 500
documents = []

# Loop through the list and slice it
for i in range(0, len(raw_tokens), chunk_size):
    chunk = raw_tokens[i:i + chunk_size]
    # Apply your Part A preprocessing here to the chunk
    # We join them back into a string because Sklearn expects strings, not lists
    clean_chunk = preprocess_text(chunk, remove_stopwords=True, use_lemmatization=True)
    documents.append(" ".join(clean_chunk))

print(f"Total number of documents created: {len(documents)}")
print(f"Sample document (first 50 chars): {documents[0][:50]}...")

Total number of documents created: 385
Sample document (first 50 chars): emma jane austen 1816 volume chapter emma woodhous...


In [24]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

# 1. Initialize Vectorizers
# min_df=2 means "ignore words that appear in less than 2 documents" (removes typos/noise)
count_vec = CountVectorizer(min_df=2)
tfidf_vec = TfidfVectorizer(min_df=2)

# 2. Fit and Transform
# This learns the vocabulary and converts text to numbers
bow_matrix = count_vec.fit_transform(documents)
tfidf_matrix = tfidf_vec.fit_transform(documents)

# 3. Report Shapes
print(f"Bag-of-Words Matrix Shape: {bow_matrix.shape}")
print(f"TF-IDF Matrix Shape:       {tfidf_matrix.shape}")
# Shape format is: (Number of Documents, Vocabulary Size)

# 4. Show Top 15 TF-IDF terms for 2 specific documents
feature_names = tfidf_vec.get_feature_names_out()

def print_top_terms(doc_id, vector_matrix, terms, n=15):
    # Get the row for the specific document
    row = vector_matrix[doc_id]

    # Convert to a list of (term_index, score)
    # .tocoo() converts sparse matrix to coordinate format (easier to iterate)
    sorted_items = sorted(zip(row.tocoo().col, row.tocoo().data), key=lambda x: -x[1])

    print(f"\n--- Top {n} Terms for Document {doc_id} ---")
    for idx, score in sorted_items[:n]:
        print(f"{terms[idx]}: {score:.4f}")

# Let's inspect Document 0 (Start of book) and Document 50 (Middle)
print_top_terms(0, tfidf_matrix, feature_names)
print_top_terms(50, tfidf_matrix, feature_names)

Bag-of-Words Matrix Shape: (385, 3890)
TF-IDF Matrix Shape:       (385, 3890)

--- Top 15 Terms for Document 0 ---
taylor: 0.3219
governess: 0.2521
sorrow: 0.1919
friend: 0.1524
wedding: 0.1488
disposition: 0.1288
miss: 0.1267
emma: 0.1212
sister: 0.1162
long: 0.1130
mother: 0.1085
father: 0.1056
daughter: 0.1016
indulgent: 0.1009
shadow: 0.1009

--- Top 15 Terms for Document 50 ---
harriet: 0.2668
convinced: 0.2092
young: 0.1686
man: 0.1662
cause: 0.1610
match: 0.1459
good: 0.1274
rationally: 0.1191
sentimentally: 0.1191
pleading: 0.1191
elton: 0.1191
vicar: 0.1132
cast: 0.1132
failure: 0.1132
prominent: 0.1132


In [25]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Compute Similarity Matrix
sim_matrix = cosine_similarity(tfidf_matrix)

# 2. Find the most similar pair (excluding diagonal self-similarity)
# We fill the diagonal with 0 so we don't find that Doc 0 is similar to Doc 0
np.fill_diagonal(sim_matrix, 0)

# Find indices of the maximum value
max_sim = sim_matrix.max()
# This returns the row/col index of the max value
max_idx = np.unravel_index(sim_matrix.argmax(), sim_matrix.shape)

doc_a, doc_b = max_idx

print(f"\nMost similar documents are #{doc_a} and #{doc_b}")
print(f"Similarity Score: {max_sim:.4f}")

# 3. Create a small table of the first 5 documents
print("\n--- Similarity Table (First 5 Docs) ---")
df_sim = pd.DataFrame(sim_matrix[:5, :5],
                      columns=[f"Doc {i}" for i in range(5)],
                      index=[f"Doc {i}" for i in range(5)])
print(df_sim)


Most similar documents are #54 and #55
Similarity Score: 0.5257

--- Similarity Table (First 5 Docs) ---
          Doc 0     Doc 1     Doc 2     Doc 3     Doc 4
Doc 0  0.000000  0.194241  0.247111  0.097382  0.125248
Doc 1  0.194241  0.000000  0.208614  0.098176  0.095089
Doc 2  0.247111  0.208614  0.000000  0.186858  0.200502
Doc 3  0.097382  0.098176  0.186858  0.000000  0.146108
Doc 4  0.125248  0.095089  0.200502  0.146108  0.000000


## Part C

Goal: Train Word2Vec embeddings and explore semantic similarity.

In [26]:
!pip install gensim
from gensim.models import Word2Vec
import multiprocessing
from nltk.corpus import gutenberg

In [27]:
# --- Part C1: Prepare training data ---

# 1. Prepare Sentences
# Word2Vec needs a list of lists of tokens: [['this', 'is', 'sent', '1'], ['sent', '2']]

# Create a list of preprocessed sentences for embeddings
# We will use the preprocess_text function, but explicitly keep stopwords.
preprocessed_sents_for_embeddings = []

# Load the sentences from the corpus
for sentence in gutenberg.sents(corpus_name):
    # Apply preprocessing, keeping stopwords but applying lemmatization
    processed_sentence = preprocess_text(sentence, remove_stopwords=False, use_lemmatization=True)
    if processed_sentence: # Only add non-empty sentences
        preprocessed_sents_for_embeddings.append(processed_sentence)

print(f"Total number of preprocessed sentences: {len(preprocessed_sents_for_embeddings)}")

print("\nSample preprocessed sentence (first 10 tokens):")
if preprocessed_sents_for_embeddings:
    print(preprocessed_sents_for_embeddings[0][:10])
else:
    print("No sentences were processed.")

# We usually do lowercase/remove punctuation and numbers here for better embeddings.
clean_sents = []
for s in preprocessed_sents_for_embeddings:
    clean_sents.append([w.lower() for w in s if w.isalpha()])
print("\nSample cleaned sentence (first 10 tokens):")
if clean_sents:
    print(clean_sents[:10])
else:
    print("No sentences were processed.")


Total number of preprocessed sentences: 7713

Sample preprocessed sentence (first 10 tokens):
['emma', 'by', 'jane', 'austen', '1816']

Sample cleaned sentence (first 10 tokens):
[['emma', 'by', 'jane', 'austen'], ['volume', 'i'], ['chapter', 'i'], ['emma', 'woodhouse', 'handsome', 'clever', 'and', 'rich', 'with', 'a', 'comfortable', 'home', 'and', 'happy', 'disposition', 'seemed', 'to', 'unite', 'some', 'of', 'the', 'best', 'blessing', 'of', 'existence', 'and', 'had', 'lived', 'nearly', 'twenty', 'one', 'year', 'in', 'the', 'world', 'with', 'very', 'little', 'to', 'distress', 'or', 'vex', 'her'], ['she', 'wa', 'the', 'youngest', 'of', 'the', 'two', 'daughter', 'of', 'a', 'most', 'affectionate', 'indulgent', 'father', 'and', 'had', 'in', 'consequence', 'of', 'her', 'sister', 's', 'marriage', 'been', 'mistress', 'of', 'his', 'house', 'from', 'a', 'very', 'early', 'period'], ['her', 'mother', 'had', 'died', 'too', 'long', 'ago', 'for', 'her', 'to', 'have', 'more', 'than', 'an', 'indistin

In [28]:
# --- Part C1: Prepare training data ---

# 1. Prepare Sentences
# Word2Vec needs a list of lists of tokens: [['this', 'is', 'sent', '1'], ['sent', '2']]
# # Load the sentences from the corpus
# sents = gutenberg.sents(corpus_name)
# # We usually do lowercase/remove punctuation here for better embeddings.
# clean_sents = []
# for s in sents:
#     clean_sents.append([w.lower() for w in s if w.isalpha()])

In [29]:
# print("\n--- Part C2: Word Embeddings (Word2Vec) ---")

# 2. Train Word2Vec
# vector_size=100: Each word will be a list of 100 numbers
# window=5: Look 5 words behind and ahead
# min_count=5: Ignore words that appear fewer than 5 times
# sg=0(for CBOW) and epochs=5, both set to default
w2v_model = Word2Vec(sentences=clean_sents,
                     vector_size=100,
                     window=5,
                     min_count=5,
                     workers=multiprocessing.cpu_count())


In [30]:
# Gotten from the most frequent tokens in A2
frequent_words = ['mr', 'emma', 'could', 'would', 'miss']

print("\n--- Part C3: Explore Similarity ---")

for word in frequent_words:
    if word in w2v_model.wv:
        print(f"\nWords most similar to '{word}':")
        similar_words = w2v_model.wv.most_similar(word, topn=10)
        for sim_word, score in similar_words:
            print(f"  {sim_word}: {score:.4f}")
    else:
        print(f"\n'{word}' not in vocabulary.")


--- Part C3: Explore Similarity ---

Words most similar to 'mr':
  john: 0.7699
  s: 0.7609
  miss: 0.7549
  frank: 0.7419
  jane: 0.6690
  necessarily: 0.6553
  handwriting: 0.6517
  glancing: 0.6422
  said: 0.6347
  vexation: 0.6332

Words most similar to 'emma':
  presently: 0.9836
  smiling: 0.9815
  colonel: 0.9812
  campbell: 0.9801
  ah: 0.9800
  henry: 0.9773
  displeasure: 0.9772
  join: 0.9770
  write: 0.9767
  telling: 0.9765

Words most similar to 'could':
  would: 0.9902
  should: 0.9734
  must: 0.9666
  did: 0.9555
  might: 0.9461
  may: 0.9254
  can: 0.9250
  ought: 0.9232
  but: 0.9227
  knew: 0.9221

Words most similar to 'would':
  could: 0.9902
  should: 0.9869
  must: 0.9802
  did: 0.9578
  can: 0.9412
  may: 0.9407
  might: 0.9381
  will: 0.9307
  ought: 0.9268
  if: 0.9245

Words most similar to 'miss':
  jane: 0.8972
  dear: 0.7556
  mr: 0.7549
  said: 0.7526
  poor: 0.7359
  my: 0.7296
  john: 0.7269
  smith: 0.7229
  latter: 0.7143
  fairfax: 0.7139


### Interpretation of Similar Words

*   **'mr'**: The similar words often include other titles ('mrs', 'miss') or names, indicating the model captures social roles and naming conventions.
*   **'emma'**: Similar words relate to emotions, actions, or states of being associated with her character (e.g., 'smiling', 'obliging', 'going'). This reflects the emotional and active nature of the protagonist in the novel.
*   **'could'** and **'would'**: These modal verbs frequently appear with other verbs or adverbs that describe possibilities, uncertainties, or desires, which is consistent with their grammatical function. The similar words show how the model groups words by their functional context.
*   **'miss'**: Similar words include other titles ('mrs', 'lady', 'jane') or names, or words related to young unmarried women, reflecting its usage in the novel as a social address and sometimes a descriptor. The high similarity with 'jane' is likely due to 'Jane Fairfax', another prominent female character often referred to as Miss Jane.

In [31]:
# 4a. Analogy (The famous King - Man + Woman = Queen)
# In Emma, maybe: husband - man + woman = wife?
try:
    analogy = w2v_model.wv.most_similar(positive=['woman', 'husband'], negative=['man'], topn=1)
    print(f"\nAnalogy (Husband - Man + Woman): {analogy[0][0]}")
except:
    print("\nCould not find analogy (vocab might be too small).")


Analogy (Husband - Man + Woman): approbation


In [32]:
# 4b. Analogy (The famous King - Man + Woman = Queen)
# father - man + woman = mother?
try:
    analogy = w2v_model.wv.most_similar(positive=['father', 'woman'], negative=['man'], topn=1)
    print(f"\nAnalogy (Father - Man + Woman): {analogy[0][0]}")
except:
    print("\nCould not find analogy (vocab might be too small).")


Analogy (Father - Man + Woman): and


In [33]:
# 4c. Analogy: King - Man + Woman = Queen
try:
    analogy = w2v_model.wv.most_similar(positive=['queen', 'man'], negative=['king'], topn=1)
    print(f"\nAnalogy (King - Man + Woman): {analogy[0][0]}")
except:
    print("\nCould not find analogy (vocab might be too small).")


Could not find analogy (vocab might be too small).


In [34]:
# 4d. Analogy: Mr - Man + Woman = Mrs
try:
    analogy = w2v_model.wv.most_similar(positive=['mrs', 'man'], negative=['mr'], topn=1)
    print(f"\nAnalogy (Mr - Man + Woman): {analogy[0][0]}")
except:
    print("\nCould not find analogy (vocab might be too small or relationship not strong enough).")


Could not find analogy (vocab might be too small or relationship not strong enough).


The analogies shown are weak because the corpus size, while a complete novel, is relatively small compared to the large datasets used to train the Word2Vec model. And a smaller corpus means fewer unique word contexts, making generalization more difficullt.

Also, the domain is very specific because it is a novel about a particular topic, so words like king and queen may not appear in here at all. And for words that do appear, they have to show up frequently enough for the model to learn stable and meaningful vectors for them.